# Generate features and a dataset from a financial time series 

This notebook serves as a practical illustration of how to leverage the functions provided in create_dataframe.py. It demonstrates step‑by‑step usage of each utility, allowing users to:
- Load raw data sources 
 - Generate the exact feature set employed in the research paper, ensuring full reproducibility of the experimental results.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from collections import Counter

from data_augmentation.decompostion import *
from data_augmentation.create_features import *

# Data reduction

In [3]:
raw_data = pd.read_csv('data/training_data.csv')
raw_data['Date'] = pd.to_datetime(raw_data['Date'], format='%Y-%m-%d')

start_train = pd.Timestamp('2010-01-05')   
end_train = pd.Timestamp('2018-12-31')  

start_val = pd.Timestamp('2019-01-01')   
end_val = pd.Timestamp('2020-06-30')  

start_test = pd.Timestamp('2020-07-01')   
end_test = pd.Timestamp('2021-12-31')  

mask_train = (raw_data['Date'] >= start_train) & (raw_data['Date'] <= end_train)
df_train = raw_data.loc[mask_train]

mask_val = (raw_data['Date'] >= start_val) & (raw_data['Date'] <= end_val)
df_val = raw_data.loc[mask_val]

mask_test = (raw_data['Date'] >= start_test) & (raw_data['Date'] <= end_test)
df_test = raw_data.loc[mask_test]

df_train.shape, df_val.shape, df_test.shape

((2263, 434), (377, 434), (380, 434))

In [4]:
raw_data.head(5)

,Date,A,AAL,AAPL,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XRAY,YUM,ZBH,ZBRA,ZION
0,2010-01-05,-0.010863,0.113207,0.001729,-0.008079,-0.003336,0.006180,0.016446,-0.001579,0.005402,...,-0.002241,0.021228,0.060819,-0.011859,0.003904,-0.011887,-0.003420,0.031656,-0.001744,0.035259
1,2010-01-06,-0.003553,-0.041431,-0.015906,0.005553,-0.004323,0.010631,-0.002122,-0.001898,-0.002528,...,0.016099,-0.011057,-0.013117,0.001921,0.008644,0.006588,-0.007149,-0.000323,-0.007687,0.086957
2,2010-01-07,-0.001296,0.029469,-0.001849,0.008284,-0.005882,-0.000935,-0.019405,-0.007921,-0.010456,...,-0.002948,-0.004472,0.021356,-0.004312,-0.003142,0.013091,-0.000288,0.022940,-0.025000,0.112000
3,2010-01-08,-0.000325,-0.019084,0.006649,0.005113,-0.001972,-0.003978,-0.005422,0.005749,-0.012488,...,-0.001108,-0.009210,-0.007164,0.000481,-0.004012,0.000000,0.000288,-0.021005,-0.003250,-0.016187
4,2010-01-11,0.000649,-0.019455,-0.008822,0.005086,-0.003106,-0.000940,-0.013083,-0.005716,0.000648,...,0.009989,0.006575,-0.003241,0.009620,0.011220,0.012921,0.017281,0.022100,0.003261,0.006094


In [5]:
X = df_train.values[:, 1:].astype(float).copy()
_,d = X.shape
window = 253

F, F_scaled, P_m, Z, mu_hat, sigma_hat, eigvals_m, eigvals = get_decomposition(X)
_, m = F_scaled.shape

F_scaled.shape, P_m.shape, Z.shape

((2263, 14), (433, 14), (2263, 433))

In [16]:
cluster_sets, labels = get_cluster(F_scaled, eigvals_m, window, nc=3)

c1 = cluster_sets[0]
c2 = cluster_sets[1]
c3 = cluster_sets[2]
c1.shape, c2.shape, c3.shape, Counter(labels)

np.save('data/sp500_cluster_1.npy', c1)
np.save('data/sp500_cluster_2.npy', c2)
np.save('data/sp500_cluster_3.npy', c3)

In [11]:
c1_synth = np.load('data/c1_synth.npy')
c2_synth = np.load('data/c2_synth.npy')
c3_synth = np.load('data/c3_synth.npy')

clusters_synth = [c1_synth, c2_synth, c3_synth]
c1_synth.shape, c2_synth.shape, c3_synth.shape

((180000, 253), (80009, 253), (20450, 253))

In [12]:
F_ = np.lib.stride_tricks.sliding_window_view(F, window_shape=253, axis=0).transpose(0,2,1)
Z_ = np.lib.stride_tricks.sliding_window_view(Z, window_shape=253, axis=0).transpose(0,2,1)

F.shape, F_.shape, Z.shape, Z_.shape

((2263, 14), (2011, 253, 14), (2263, 433), (2011, 253, 433))

In [14]:
F_synth = get_F_synth(clusters_synth, labels, eigvals_m, window, m, 2000)
Z_fitted = fit_Z_gmm(Z)
Z_synth = get_Z_synth_gmm(Z_fitted, F_synth, d)
F_synth.shape, Z_synth.shape

Sampling from 2‑Gaussian GMM per asset: 100%|█| 433/433 [00:13<00:00, 31.20it/s]


((2000, 253, 14), (2000, 253, 433))

In [15]:
l=len(Z_synth)
dim = Z_synth.shape[-1]
return_synth_sbb = reconstruct_X(F_synth[:l, :, :dim], Z_synth[:l, :, :], P_m[:dim], mu_hat[:dim], sigma_hat[:dim])
return_synth_sbb.shape

(2000, 253, 433)

# Generate the features 

## If your raw data contains instrument name :

Often used to generate Tabular Data from a real-world time series like S&P returns

In [12]:
raw_data = pd.read_csv('data/training_data.csv')
sp, matrix = load_real_data(date_col = 'Date', path = 'data/training_data.csv')
sp

,perimeter.DateTime,perimeter.instr_id,target.target
0,2010-01-05,A,-0.010863
3020,2010-01-05,AAL,0.113207
6040,2010-01-05,AAPL,0.001729
9060,2010-01-05,ABT,-0.008079
12080,2010-01-05,ACGL,-0.003336
...,...,...,...
1295579,2021-12-31,XRAY,-0.007295
1298599,2021-12-31,YUM,0.003396
1301619,2021-12-31,ZBH,-0.009589
1304639,2021-12-31,ZBRA,-0.003916


In [6]:
df = make_real_dataframe(sp_returns = sp, 
                         returns_matrix = matrix,
                         cum_horizons = (5, 10, 21, 63, 126, 252),
                         vol_horizons = (5, 10, 21, 63, 126, 252),
                         lag_zscore_horizons = (3, 5, 10, 21),
                         lag_market_return_horizons = (1, ),
                        ) 
df.sample(6)

,perimeter.DateTime,perimeter.path_id,perimeter.instr_name,perimeter.Date,perimeter.instr_id,feature.cum_ret_5d,feature.cum_ret_10d,feature.cum_ret_21d,feature.cum_ret_63d,feature.cum_ret_126d,...,feature.mkt_vol_3d,feature.mkt_vol_5d,feature.mkt_vol_10d,feature.mkt_vol_21d,feature.mkt_mean_3d,feature.mkt_mean_5d,feature.mkt_mean_10d,feature.mkt_mean_21d,feature.return_t-1_market,target.target
1157475,2020-08-18,REAL SP500,BXP,2674,66,-0.052436,0.001251,-0.072534,0.102980,-0.389749,...,0.002471,0.004241,0.005089,0.006487,-0.000914,0.000428,0.003565,0.002372,0.000035,-0.007304
1048949,2019-08-20,REAL SP500,KIM,2423,223,-0.002151,0.009615,0.057670,0.072310,0.102837,...,0.005911,0.016533,0.014465,0.012755,0.009614,0.002455,0.002704,-0.000644,0.010945,-0.025370
1174452,2020-10-13,REAL SP500,FAST,2713,156,0.039468,0.041289,0.080857,0.106869,0.351252,...,0.004649,0.009379,0.008800,0.011823,0.007930,0.006476,0.005634,0.002818,0.007579,-0.048069
624182,2015-09-25,REAL SP500,L,1442,229,-0.015070,-0.014151,0.013098,-0.076055,-0.113663,...,0.005555,0.007678,0.008977,0.014844,-0.007012,-0.006681,-0.001103,0.001594,-0.003638,0.006999
1231358,2021-04-22,REAL SP500,ROP,2844,339,0.025250,0.035647,0.070567,0.029694,0.031373,...,0.008198,0.007105,0.005287,0.007273,0.000183,0.002605,0.002240,0.002506,0.011681,0.003445
1198585,2021-01-04,REAL SP500,AWK,2769,41,0.040563,0.020851,-0.003870,0.040882,0.175812,...,0.005595,0.004355,0.004967,0.006186,0.003106,0.002774,0.001295,0.001411,0.007828,-0.021046


## If your raw data does not contain instrument name :

Often used to generate Tabular Data from a synthetic generated time series 

In [9]:
returns_synth = np.load('data/returns_synth.npy')
print(f"returns_synth shape is {returns_synth.shape} with {returns_synth.shape[0]} "
f"the number of paths generated, {returns_synth.shape[1]} the number of days and {returns_synth.shape[1]} instruments")
#returns_synth[0]

returns_synth shape is (10, 253, 433) with 10 the number of paths generated, 253 the number of days and 253 instruments


In [10]:
path_id = 8
df = make_path_dataframe(path_returns=returns_synth[path_id], 
                         path_id = path_id, 
                         cum_horizons = (5, 10, 21, 63, 126, 252),
                         vol_horizons = (5, 10, 21, 63, 126, 252),
                         lag_zscore_horizons = (3, 5, 10, 21),
                         lag_market_return_horizons = (1,)
                        )
df.sample(6)

,perimeter.path_id,perimeter.Date,perimeter.instr_id,feature.cum_ret_5d,feature.cum_ret_10d,feature.cum_ret_21d,feature.cum_ret_63d,feature.cum_ret_126d,feature.cum_ret_252d,feature.vol_5d,...,feature.mkt_vol_5d,feature.mkt_vol_10d,feature.mkt_vol_21d,feature.mkt_mean_3d,feature.mkt_mean_5d,feature.mkt_mean_10d,feature.mkt_mean_21d,feature.return_t-1_market,target.target,extra.Return
101842,8,236,87,-0.034427,-0.023356,0.068122,0.130022,0.192723,NaN,0.034446,...,0.010774,0.009245,0.010251,-0.008834,-0.001658,-0.002920,-0.000201,-0.013104,1,0.034529
29739,8,69,295,0.072404,0.084135,0.072414,0.072819,NaN,NaN,0.008232,...,0.003067,0.004321,0.006192,0.005101,0.004659,0.004570,0.003131,0.005266,1,0.003604
71084,8,165,72,-0.024979,-0.048899,-0.101558,-0.033108,0.226705,NaN,0.014768,...,0.004640,0.004931,0.004495,0.002165,-0.000394,0.001364,0.003237,0.003076,1,0.044124
105349,8,244,130,0.018349,-0.017742,-0.004964,-0.040839,0.427477,NaN,0.017015,...,0.003300,0.007197,0.008950,-0.001976,-0.002159,0.000472,0.000428,-0.005766,0,-0.027205
40957,8,95,255,0.034077,0.059547,0.166992,0.494650,NaN,NaN,0.020897,...,0.011633,0.008725,0.007856,0.000941,-0.000821,-0.000272,0.000973,-0.005846,1,0.021084
55230,8,128,239,0.010822,0.049906,-0.002946,0.202227,0.283137,NaN,0.009861,...,0.007027,0.006008,0.006590,0.006484,0.003007,0.002467,-0.000543,0.009927,0,-0.017116
